# CAS Exam 5: Allocated Loss Adjustment Expense (ALAE) Estimation

**Source:** Friedland, J. *Estimating Unpaid Claims Using Basic Techniques*, Casualty Actuarial Society, 2010 — Chapter 16

**Exam task covered:** B14 — Calculate and evaluate estimation techniques for allocated loss adjustment expenses

**Learning goals:**
1. Distinguish ALAE from ULAE and understand why ALAE is treated separately from losses
2. Apply the paid ALAE to paid loss ratio method
3. Apply the paid ALAE to paid (loss + ALAE) ratio method
4. Apply the development method directly to an ALAE triangle
5. Apply the Bornhuetter-Ferguson method to ALAE
6. Understand when to use each method and what drives ALAE trends

## Formula Sheet Quick Reference

| Method | Formula |
|---|---|
| **ALAE / Loss ratio** | $r = \text{Paid ALAE} / \text{Paid Loss}$; select $r$; $\text{Ult ALAE} = r \times \text{Ult Loss}$ |
| **ALAE / (Loss+ALAE) ratio** | $s = \text{Paid ALAE} / (\text{Paid Loss}+\text{Paid ALAE})$; $\text{Ult ALAE} = \frac{s}{1-s} \times \text{Ult Loss}$ |
| **Development method** | Develop ALAE triangle like loss triangle; $\text{Ult ALAE} = \text{Latest ALAE} \times \text{CDF}_{\text{ALAE}}$ |
| **BF on ALAE** | $\text{Ult ALAE} = \text{Paid ALAE} + \%\text{Unreported} \times \text{Expected ALAE}$ |
| **Expected ALAE (BF)** | $\text{Expected ALAE} = \text{Expected ALAE ratio} \times \text{Ult Loss}$ |
| **ALAE IBNR** | $\text{ALAE IBNR} = \text{Ult ALAE} - \text{Latest Paid ALAE}$ |

### Relationship Between Ratio Methods

If $r = \text{ALAE/Loss}$ and $s = \text{ALAE/(Loss+ALAE)}$, then:

$$r = \frac{s}{1-s} \qquad \text{and} \qquad s = \frac{r}{1+r}$$

## ALAE Overview

### What is ALAE?

**Allocated Loss Adjustment Expenses (ALAE)** are costs directly attributable to a specific claim that are separately tracked in the claim file. Unlike losses themselves, ALAE do not compensate the claimant — they are the insurer's cost of managing the claim.

**Common ALAE items:**
- Outside defense attorney fees
- Court costs and filing fees
- Expert witness fees (accident reconstruction, medical experts)
- Independent medical examinations (IMEs)
- Private investigator costs
- Structured settlement costs

**ALAE vs. ULAE:**

| Feature | ALAE | ULAE |
|---|---|---|
| Assignable to specific claim | Yes — tracked in the claim file | No — general overhead |
| Examples | Defense counsel, court costs | Adjuster salaries, claims dept overhead |
| NAIC equivalent | DCC (Defense & Cost Containment) + part of A&O | Part of A&O |
| Development pattern | Similar to losses; may develop further | Estimated separately via ratio method |

---

### Why Estimate ALAE Separately from Losses?

1. **ALAE trends differently from losses.** Attorney involvement, court costs, and defense fees inflate at different rates than underlying claim severity.
2. **ALAE is disproportionate for large claims.** Defense costs on a $2M liability claim are far more than double the costs of a $1M claim; complex litigation is not proportional.
3. **ALAE ratios differ by line.** Bodily injury auto has high ALAE (defense counsel is routine); auto physical damage has essentially zero ALAE.
4. **ALAE develops more predictably.** Defense costs are typically incurred before the claim settles, so ALAE development at early ages is relatively high.

---

### Key Drivers of ALAE

| Driver | Effect |
|---|---|
| **Attorney involvement rate** | More represented claims → higher ALAE ratio |
| **Litigation complexity** | Multi-party suits, expert-intensive cases → higher per-claim ALAE |
| **Defense strategy** | Aggressive defense → higher ALAE; early settlement → lower ALAE |
| **Line of business** | Professional liability, D&O, GL → high ALAE; property, ADP → low ALAE |
| **Claim size** | ALAE/Loss ratio is typically *higher* for large claims (leveraged effect) |

> **Exam tip** — A key ALAE topic: if you are told that attorney involvement is increasing, the ALAE ratio will likely trend upward. Conversely, if the insurer implements a "settle-early" claims strategy, the ALAE ratio may decline. Either change makes a simple historical-average ALAE ratio unreliable as a prospective estimate.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda x: f'{x:,.0f}')

# ─── Sample data: latest-diagonal paid losses and paid ALAE by AY ─────────────
# Represents mature, fully-developed values (or "ultimate" for older AYs)

data = pd.DataFrame({
    'Paid_Loss':    [8_200, 9_100, 10_300, 11_800, 13_200, 5_600],  # last AY is immature
    'Paid_ALAE':    [  820,   950,  1_100,  1_298,  1_452,   420],  # ($000s)
    'Ultimate_Loss': [8_200, 9_100, 10_300, 11_800, 13_200, 14_500],  # known for older AYs
}, index=[2018, 2019, 2020, 2021, 2022, 2023])

data['ALAE_Loss_Ratio']     = data['Paid_ALAE'] / data['Paid_Loss']
data['ALAE_LossALAE_Ratio'] = data['Paid_ALAE'] / (data['Paid_Loss'] + data['Paid_ALAE'])

print('Historical paid ALAE and ratio data:')
print()
pd.set_option('display.float_format', lambda x: f'{x:,.4f}' if abs(x) < 10 else f'{x:,.0f}')
print(data.to_string())

## Method 1 — Paid ALAE to Paid Loss Ratio

### Concept

The simplest and most commonly used ALAE method. Compute the historical ratio of paid ALAE to paid losses at the latest evaluation, select a representative ratio, and apply it to the projected ultimate losses.

$$r = \frac{\text{Paid ALAE}}{\text{Paid Loss}} \qquad \text{(historical, per AY)}$$

$$\text{Ultimate ALAE} = r_{\text{selected}} \times \text{Ultimate Loss}$$

$$\text{ALAE IBNR} = \text{Ultimate ALAE} - \text{Paid ALAE}$$

### When to Use
- Stable historical ALAE/loss ratios with no clear trend
- No major changes in claims handling or litigation strategy
- The selected ratio is from mature accident years where ALAE is fully developed

### When NOT to Use
- ALAE ratios are trending up or down over time
- Recent changes in defense strategy or attorney involvement rates
- ALAE is concentrated in a few very large claims (high volatility in the ratio)

> **Exam tip** — For immature accident years, the latest ALAE/loss ratio understates the ultimate ratio because ALAE tends to be incurred later in the claim's life (after discovery, just before trial). Always use mature years to calibrate the ratio, or explicitly trend the ratio to the future period.

In [ ]:
# ─── Method 1: Paid ALAE / Paid Loss Ratio ────────────────────────────────────
pd.set_option('display.float_format', lambda x: f'{x:,.0f}')

# Use mature AYs (2018-2022) to select ratio; exclude 2023 (immature)
mature = data.loc[2018:2022]

ratio_by_ay = mature['Paid_ALAE'] / mature['Paid_Loss']

print('ALAE/Loss ratio by accident year (mature):')
print(ratio_by_ay.map('{:.4f}'.format).to_frame('ALAE/Loss Ratio'))
print()

# Examine trend: is the ratio increasing?
slope = np.polyfit(range(len(ratio_by_ay)), ratio_by_ay.values, 1)[0]
print(f'Trend slope: {slope:.5f} per year')
if slope > 0.001:
    print('  -> Ratio is trending upward; consider weighting recent years more heavily')
elif slope < -0.001:
    print('  -> Ratio is trending downward; investigate claims handling changes')
else:
    print('  -> Ratio appears stable; simple average is appropriate')

print()
selected_ratio = ratio_by_ay.iloc[-3:].mean()  # 3-year simple average
print(f'Selected ALAE/Loss ratio (3-year average): {selected_ratio:.4f}')

In [ ]:
# ─── Apply ratio to project ALAE ─────────────────────────────────────────────
results_m1 = data[['Paid_Loss', 'Paid_ALAE', 'Ultimate_Loss']].copy()
results_m1['Ult_ALAE_M1']  = results_m1['Ultimate_Loss'] * selected_ratio
results_m1['ALAE_IBNR_M1'] = results_m1['Ult_ALAE_M1'] - results_m1['Paid_ALAE']

print(f'Method 1 (ALAE/Loss = {selected_ratio:.4f}) — ALAE Projection:')
print(results_m1[['Paid_Loss', 'Paid_ALAE', 'Ultimate_Loss',
                   'Ult_ALAE_M1', 'ALAE_IBNR_M1']].to_string())
print()
print(f'Total ALAE IBNR (Method 1): {results_m1["ALAE_IBNR_M1"].sum():>10,.0f}')
print(f'Total ALAE Ultimate:        {results_m1["Ult_ALAE_M1"].sum():>10,.0f}')

## Method 2 — Paid ALAE to Paid (Loss + ALAE) Ratio

### Concept

A refinement of Method 1. Instead of the ratio to losses alone, use the ratio of ALAE to total incurred (losses + ALAE). This is sometimes preferred because:
- It bounds the ratio between 0% and 100% naturally
- It has a straightforward actuarial interpretation: ALAE as a fraction of all claim costs

$$s = \frac{\text{Paid ALAE}}{\text{Paid Loss} + \text{Paid ALAE}}$$

**Converting to an ultimate ALAE estimate:**

$$\text{Ultimate ALAE} = \frac{s}{1-s} \times \text{Ultimate Loss}$$

This is algebraically identical to Method 1 with $r = s/(1-s)$.

### Relationship to Method 1

If the ALAE/Loss ratio is $r = 12\%$, then $s = 12\%/(1+12\%) = 10.71\%$.  
Both methods give exactly the same ALAE ultimate when applied consistently.

In [ ]:
# ─── Method 2: ALAE / (Loss + ALAE) Ratio ─────────────────────────────────────
s_ratios = mature['Paid_ALAE'] / (mature['Paid_Loss'] + mature['Paid_ALAE'])
selected_s = s_ratios.iloc[-3:].mean()

print('ALAE/(Loss+ALAE) ratio by accident year (mature):')
print(s_ratios.map('{:.4f}'.format).to_frame('s ratio'))
print(f'\nSelected s ratio (3-year average): {selected_s:.4f}')
print(f'Equivalent ALAE/Loss ratio r = s/(1-s): {selected_s/(1-selected_s):.4f}')
print(f'Method 1 selected ratio was: {selected_ratio:.4f}')
print()

results_m2 = data[['Paid_Loss', 'Paid_ALAE', 'Ultimate_Loss']].copy()
results_m2['Ult_ALAE_M2']  = results_m2['Ultimate_Loss'] * (selected_s / (1 - selected_s))
results_m2['ALAE_IBNR_M2'] = results_m2['Ult_ALAE_M2'] - results_m2['Paid_ALAE']

print(f'Method 2 (s = {selected_s:.4f}) — ALAE Projection:')
print(results_m2[['Paid_Loss', 'Paid_ALAE', 'Ultimate_Loss',
                   'Ult_ALAE_M2', 'ALAE_IBNR_M2']].to_string())
print()
print(f'Total ALAE IBNR (Method 2): {results_m2["ALAE_IBNR_M2"].sum():>10,.0f}')

## Method 3 — Development Method on ALAE Triangle

### Concept

Treat the ALAE triangle exactly like a loss triangle and apply standard chain ladder development. Build an ALAE-only triangle (paid ALAE by accident year and development age), compute link ratios, select development factors, and project to ultimate ALAE directly.

$$\text{Ultimate ALAE} = \text{Latest Paid ALAE} \times \text{CDF}_{\text{ALAE}}(d \to \text{ult})$$

### When to Use
- ALAE data is available in a triangle format by accident year and development age
- ALAE development pattern is stable across accident years
- ALAE is large enough to analyze separately (smaller ALAE may not have credible triangle data)

### Relationship to Loss Triangle Development

- ALAE LDFs may differ from loss LDFs: defense costs are often incurred throughout the claim's life (during discovery, depositions), while large loss payments may be more back-loaded
- For lines with significant defense costs (GL, WC, professional liability), ALAE can develop substantially beyond paid losses
- For short-tailed lines or property claims, ALAE and losses typically develop together

In [ ]:
# ─── Method 3: ALAE Development Triangle ─────────────────────────────────────
pd.set_option('display.float_format', lambda x: f'{x:,.0f}')

# Hypothetical cumulative paid ALAE triangle ($000s)
alae_triangle = pd.DataFrame({
    '12':  [ 280,  320,  350,  380,  420,  165],
    '24':  [ 540,  610,  680,  740,  np.nan, np.nan],
    '36':  [ 700,  790,  870,  np.nan, np.nan, np.nan],
    '48':  [ 790,  890,  np.nan, np.nan, np.nan, np.nan],
    '60':  [ 820,  np.nan, np.nan, np.nan, np.nan, np.nan],
}, index=[2018, 2019, 2020, 2021, 2022, 2023])

print('Cumulative paid ALAE triangle ($000s):')
print(alae_triangle.to_string())

# Compute volume-weighted LDFs for ALAE triangle
ages_alae = ['12', '24', '36', '48', '60']
alae_ldfs = {}
for i in range(len(ages_alae) - 1):
    f, t = ages_alae[i], ages_alae[i+1]
    mask = alae_triangle[[f, t]].notna().all(axis=1)
    if mask.sum() > 0:
        num = alae_triangle.loc[mask, t].sum()
        den = alae_triangle.loc[mask, f].sum()
        alae_ldfs[f'{f}-{t}'] = num / den

# Tail factor (assume development is complete by 60 months)
tail_factor = 1.010  # 1% tail beyond 60 months

print('\nVolume-weighted ALAE LDFs:')
pd.Series(alae_ldfs).map('{:.4f}'.format).to_frame('ALAE LDF')

In [ ]:
# Build CDFs from each age to ultimate
ldf_list = list(alae_ldfs.values())
age_labels = list(alae_ldfs.keys())

# CDF from age d = product of LDFs from d onward × tail
cdfs_alae = {}
running = tail_factor
cdfs_alae['60'] = tail_factor
for ldf_val, transition in zip(reversed(ldf_list), reversed(age_labels)):
    running = ldf_val * running
    from_age = transition.split('-')[0]
    cdfs_alae[from_age] = running

print('ALAE CDFs to ultimate (including tail):')
pd.Series(cdfs_alae).map('{:.4f}'.format).to_frame('CDF to Ult')

In [ ]:
# Apply CDFs to latest ALAE to project ultimate
latest_alae_age = {
    2018: ('60', alae_triangle.loc[2018, '60']),
    2019: ('48', alae_triangle.loc[2019, '48']),
    2020: ('36', alae_triangle.loc[2020, '36']),
    2021: ('24', alae_triangle.loc[2021, '24']),
    2022: ('12', alae_triangle.loc[2022, '12']),
    2023: ('12', alae_triangle.loc[2023, '12']),
}

rows_m3 = []
for ay, (age, latest) in latest_alae_age.items():
    cdf = cdfs_alae.get(age, 1.0)
    ult = latest * cdf
    ibnr = ult - latest
    rows_m3.append({'AY': ay, 'Latest ALAE': latest, 'Age': age, 'CDF': cdf,
                    'Ult ALAE (M3)': ult, 'ALAE IBNR (M3)': ibnr})

m3_df = pd.DataFrame(rows_m3).set_index('AY')
print('Method 3 (ALAE Development Method) — ALAE Projection:')
print(m3_df.to_string())
print()
print(f'Total ALAE IBNR (Method 3): {m3_df["ALAE IBNR (M3)"].sum():>10,.0f}')
print(f'Total ALAE Ultimate:        {m3_df["Ult ALAE (M3)"].sum():>10,.0f}')

## Method 4 — Bornhuetter-Ferguson Applied to ALAE

### Concept

The BF method can be applied to ALAE just as it is applied to losses. It blends the actual paid ALAE to date with an a priori estimate of expected ALAE, weighted by the development pattern.

$$\text{Ult ALAE}_{\text{BF}} = \text{Paid ALAE} + \%\text{Unreported}_{\text{ALAE}} \times \text{Expected ALAE}$$

where:
$$\%\text{Unreported}_{\text{ALAE}} = 1 - \frac{1}{\text{CDF}_{\text{ALAE}}}$$

$$\text{Expected ALAE} = \text{Expected ALAE Ratio} \times \text{Ultimate Loss}$$

The expected ALAE ratio can be set using:
- Historical mature ALAE/Loss ratios from Method 1
- Industry benchmarks
- Actuarial judgment based on line of business

### When to Use BF on ALAE
- Most useful for immature accident years where paid ALAE is too low to drive the projection
- When the ALAE development pattern is volatile (high leverage)
- As a complement/check to the development method

In [ ]:
# ─── Method 4: BF Applied to ALAE ────────────────────────────────────────────
# Use the ALAE CDFs from Method 3 and expected ALAE ratio from Method 1

expected_alae_ratio = selected_ratio  # from Method 1: historical ALAE/Loss ratio

print(f'Expected ALAE/Loss ratio (a priori): {expected_alae_ratio:.4f}')
print()

# Get ultimate losses for each AY
ult_losses = data['Ultimate_Loss'].to_dict()

rows_m4 = []
for ay, (age, paid_alae) in latest_alae_age.items():
    cdf = cdfs_alae.get(age, 1.0)
    pct_reported = 1.0 / cdf
    pct_unreported = 1.0 - pct_reported

    ult_loss = ult_losses[ay]
    expected_alae = expected_alae_ratio * ult_loss

    ult_alae_bf = paid_alae + pct_unreported * expected_alae
    alae_ibnr_bf = ult_alae_bf - paid_alae

    rows_m4.append({
        'AY': ay,
        'Paid ALAE': paid_alae,
        '% Rptd': pct_reported,
        '% Unrptd': pct_unreported,
        'Ult Loss': ult_loss,
        'Exp ALAE': expected_alae,
        'Ult ALAE (BF)': ult_alae_bf,
        'ALAE IBNR (BF)': alae_ibnr_bf,
    })

m4_df = pd.DataFrame(rows_m4).set_index('AY')
pd.set_option('display.float_format', lambda x: f'{x:,.3f}' if abs(x) < 5 else f'{x:,.0f}')
print('Method 4 (BF Applied to ALAE) — ALAE Projection:')
print(m4_df.to_string())
print()
pd.set_option('display.float_format', lambda x: f'{x:,.0f}')
print(f'Total ALAE IBNR (BF):   {m4_df["ALAE IBNR (BF)"].sum():>10,.0f}')
print(f'Total ALAE Ultimate:    {m4_df["Ult ALAE (BF)"].sum():>10,.0f}')

## Method Comparison and Selection

### Summary of All Methods

In [ ]:
# ─── Method Comparison ────────────────────────────────────────────────────────
pd.set_option('display.float_format', lambda x: f'{x:,.0f}')

comparison = pd.DataFrame({
    'Paid ALAE': data['Paid_ALAE'],
    'Method 1 (ALAE/Loss)': results_m1['Ult_ALAE_M1'],
    'Method 2 (ALAE/(L+A))': results_m2['Ult_ALAE_M2'],
    'Method 3 (Dev Method)': m3_df['Ult ALAE (M3)'],
    'Method 4 (BF on ALAE)': m4_df['Ult ALAE (BF)'],
})

print('ALAE Ultimate by Method:')
print(comparison.to_string())
print()
print('Totals:')
print(comparison.sum().to_string())
print()
print('ALAE IBNR (= Ultimate - Paid ALAE):')
ibnr_comparison = comparison.copy()
for col in ibnr_comparison.columns[1:]:
    ibnr_comparison[col] = ibnr_comparison[col] - ibnr_comparison['Paid ALAE']
ibnr_comparison = ibnr_comparison.drop(columns=['Paid ALAE'])
print(ibnr_comparison.to_string())
print()
print('Total ALAE IBNR by method:')
print(ibnr_comparison.sum().to_string())

### Selection Criteria — Which Method to Use?

| Method | Best when... | Avoid when... |
|---|---|---|
| **Method 1 (ALAE/Loss)** | Stable ratio, mature data, simple application | Trending ratio, large claim concentration |
| **Method 2 (ALAE/(L+A))** | Ratio is near 50% (avoids distortion); same pros as M1 | Same cons as M1 |
| **Method 3 (Dev Method)** | Credible ALAE triangle exists; ALAE pattern differs from loss pattern | Insufficient ALAE triangle data; small books of business |
| **Method 4 (BF on ALAE)** | Immature AYs; ALAE development is leveraged (high CDF); need to blend data with expectation | Mature AYs where paid ALAE is fully credible |

### In Practice

Actuaries typically:
1. Compute Method 1 or Method 2 as a quick check
2. Compute Method 3 if ALAE triangle data exists
3. Use Method 4 for the most recent, immature accident years
4. Select results based on credibility of the triangle data and stability of the ratio

> **Exam tip** — On the exam, you will usually be given the ALAE data in a specific format (a ratio, a triangle, or just paid ALAE and ultimate losses) and asked to calculate ALAE IBNR using a specified method. Master the formulas for all four methods and practice converting between the ALAE/Loss ratio and the ALAE/(Loss+ALAE) ratio.

---

## Practice Problems

**Problem 1 — Method 1**

An actuary has the following data for a workers compensation book ($000s):

| AY | Paid Loss | Paid ALAE | Ultimate Loss |
|---|---|---|---|
| 2020 | 12,400 | 1,488 | 12,400 |
| 2021 | 13,800 | 1,683 | 13,800 |
| 2022 | 15,200 | 1,901 | 15,200 |
| 2023 | 7,600 | 805 | 17,500 |

AYs 2020–2022 are fully developed. AY 2023 is immature. Using the 3-year average ALAE/Loss ratio from mature years, calculate ALAE IBNR for AY 2023.

*Answer:*
- Ratios: 1,488/12,400 = 12.00%; 1,683/13,800 = 12.20%; 1,901/15,200 = 12.51%
- Selected ratio (3-year average) = (12.00% + 12.20% + 12.51%) / 3 = **12.24%**
- Ultimate ALAE (AY 2023) = 12.24% × 17,500 = **$2,142**
- ALAE IBNR (AY 2023) = 2,142 − 805 = **$1,337**

---

**Problem 2 — Ratio Conversion**

For an accident year where ALAE/Loss = 15.0%, compute the ALAE/(Loss+ALAE) ratio.

*Answer:*
- $s = r / (1 + r) = 0.150 / (1 + 0.150) = 0.150 / 1.150 = \mathbf{13.04\%}$

Verification: if Ultimate Loss = $10,000 and ALAE = $1,500, then ALAE/(Loss+ALAE) = $1,500/$11,500 = 13.04% ✓

---

**Problem 3 — BF Method**

An actuary is estimating ALAE for an immature accident year:
- Paid ALAE to date: $320,000
- ALAE CDF (latest age to ultimate): 2.50
- Expected ALAE ratio (a priori): 12%
- Ultimate losses: $8,000,000

Calculate ultimate ALAE and ALAE IBNR using the BF method.

*Answer:*
- % Reported = 1/CDF = 1/2.50 = 40.0%
- % Unreported = 1 − 40.0% = **60.0%**
- Expected ALAE = 12% × $8,000,000 = $960,000
- Ultimate ALAE (BF) = $320,000 + 60.0% × $960,000 = $320,000 + $576,000 = **$896,000**
- ALAE IBNR = $896,000 − $320,000 = **$576,000**

Note: the development method would give $320,000 × 2.50 = $800,000, which is lower. The BF method produces a higher estimate here because the a priori suggests more ALAE than the emerging data alone.